# Chapter 17 – Recursive algorithms: sum, Merge Sort, binary search, maximum subarray (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_python.ipynb)

The algorithms with pseudo-code of the chapter, run on the examples of the notes: the **sum of an array** (iterative, by halving and by decrease by one), **Merge Sort** with `Merge`, the **recursive binary search** and the **maximum subarray** problem (brute force, divide and conquer, Kadane's algorithm). The output reproduces the traces and the numbers of the chapter (with indices from 0).

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

Run each code cell with **Shift+Enter**, in order from top to bottom. Python 3, no external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursive_sum` (snake_case).

## 1. The sum of an array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: a loop that accumulates the partial sums. $S(n)=3n+4$, extra memory $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide and conquer on the two halves (base case $l=r$). $2n-1$ calls, $S(n)=7n-5$, depth $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decrease by one: $S(n)=3n-1$, depth $n$.

*Input*: an array of $n\ge1$ integers (for `RecursiveSum`, two indices $l\le r$). *Output*: the sum of the elements. The counters `steps` (lines of pseudo-code executed), `calls` and `maxDepth` (simultaneously active calls) measure the quantities analysed in the chapter; in `RecursiveSum` and `RecursiveSumLinear` there is a single `return` so that the depth can be updated.

On the array of the example $a=[9,8,12,3,5,14,6]$ ($n=7$) the sum is $57$; `RecursiveSum` makes $13$ calls and $44$ steps, with depth $4$. Then we run the trace of the table of the chapter on $a=[9,8,12,3,5]$: the column *stack* reports the active calls.

In [1]:
steps = 0       # steps executed (lines of pseudo-code)
calls = 0       # calls executed
depth = 0       # calls active right now
max_depth = 0   # maximum depth of the recursion

def reset():
    global steps, calls, depth, max_depth
    steps = calls = depth = max_depth = 0

def sum_iterative(a, n):
    """Iterative sum of a[0..n-1]."""
    global steps
    s = 0
    i = 0
    steps += 2                  # lines 1-2: s := 0, i := 1
    while i < n:
        steps += 3              # true test + two lines of the body
        s = s + a[i]
        i = i + 1
    steps += 2                  # false test + return
    return s

def recursive_sum(a, l, r):
    """Sum of a[l..r] (0-based indices), l <= r, by divide and conquer."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # line 1: if l = r
    if l == r:
        steps += 1              # line 2: return a[l] (base case)
        result = a[l]
    else:
        steps += 4              # lines 3-6: m, two calls, return
        m = l + (r - l) // 2    # divide
        s1 = recursive_sum(a, l, m)         # conquer (left half)
        s2 = recursive_sum(a, m + 1, r)     # conquer (right half)
        result = s1 + s2                    # combine
    depth -= 1
    return result

def recursive_sum_linear(a, n):
    """Sum of a[0..n-1] by decrease by one, n >= 1."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # line 1: if n = 1
    if n == 1:
        steps += 1              # line 2: return a[1]
        result = a[0]
    else:
        steps += 2              # lines 3-4: call and return
        s = recursive_sum_linear(a, n - 1)
        result = s + a[n - 1]
    depth -= 1
    return result

# ---- example of the chapter ----
a = [9, 8, 12, 3, 5, 14, 6]
n = len(a)
print("a =", a, "  n =", n)
reset(); s = sum_iterative(a, n)
print(f"SumIterative       = {s}   steps={steps} (3n+4={3 * n + 4})")
reset(); s = recursive_sum(a, 0, n - 1)
print(f"RecursiveSum       = {s}   calls={calls} (2n-1={2 * n - 1})  steps={steps} (7n-5={7 * n - 5})  maxDepth={max_depth}")
reset(); s = recursive_sum_linear(a, n)
print(f"RecursiveSumLinear = {s}   calls={calls}  steps={steps} (3n-1={3 * n - 1})  maxDepth={max_depth}")

# ---- trace of RecursiveSum ----
row = 0
max_stack = 0

def emit(stack, action):
    global row, max_stack
    row += 1
    max_stack = max(max_stack, len(stack))
    print(f"{row:>2d} | {','.join(stack)} | {action}")

def recursive_sum_trace(a, l, r, stack):
    stack.append(f"({l},{r})")
    if l == r:
        emit(stack, f"l=r => return a[{l}]={a[l]}")
        stack.pop()
        return a[l]
    m = l + (r - l) // 2
    emit(stack, f"l!=r, m={l}+({r}-{l})/2={m}; call ({l},{m})")
    s1 = recursive_sum_trace(a, l, m, stack)
    emit(stack, f"s1={s1}; call ({m + 1},{r})")
    s2 = recursive_sum_trace(a, m + 1, r, stack)
    emit(stack, f"s2={s2}; return {s1}+{s2}={s1 + s2}")
    stack.pop()
    return s1 + s2

b = [9, 8, 12, 3, 5]
print()
print("a =", b, "  RecursiveSum(a, 0, 4)")
print(" # | stack | action")
res = recursive_sum_trace(b, 0, len(b) - 1, [])
print(f"result {res}, rows {row}, maximum number of active calls {max_stack}")

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | action
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Try it yourself.** Change the array `a` (also its length) and compare steps, calls and depth: only `RecursiveSumLinear` has depth $n$.

In [2]:
# Change the array: compare steps, calls and depth of the three algorithms.
a = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
n = len(a)
reset(); print(sum_iterative(a, n), steps)
reset(); print(recursive_sum(a, 0, n - 1), steps, calls, max_depth)
reset(); print(recursive_sum_linear(a, n), steps, calls, max_depth)

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort and Merge

- **MergeSort**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sub-array $a[l..r]$ sorted in non-decreasing order. It is called `MergeSort(a, 1, n)` in the pseudo-code, `merge_sort(a, 0, n-1)` in the code.
- **Merge**$(a,l,m,r)$ — *Input*: an array and indices $l\le m<r$ such that $a[l..m]$ and $a[m+1..r]$ are sorted. *Output*: $a[l..r]$ sorted. It copies the two halves into $L$ and $R$ and moves back into $a$, step by step, the smaller of the first two elements not yet copied.

The counter `comparisons` counts the comparisons between elements executed by `Merge`. On the array of the example $a=[9,12,3,16,19,11,12]$ we print the calls of `Merge` in execution order (with the array after each of them, table of the chapter; indices are from 0: adding 1 gives those of the chapter), then the trace of `Merge(a, 0, 1, 3)` on $a=[5,8,2,7]$ (example of the C++ implementation).

In [3]:
comparisons = 0   # comparisons between elements executed by Merge
ms_calls = 0      # calls of MergeSort

def merge(a, l, m, r):
    """Merge(a, l, m, r): merges the sorted a[l..m] and a[m+1..r]."""
    global comparisons
    n1 = m - l + 1                  # size of the left part
    n2 = r - m                      # size of the right part
    L = [0] * n1                    # auxiliary arrays
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:        # merge L and R into a[l..r]
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:                   # copy the rest of L
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:                   # copy the rest of R
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    """MergeSort(a, l, r): sorts a[l..r]."""
    global ms_calls
    ms_calls += 1
    if l < r:
        m = l + (r - l) // 2        # divide
        merge_sort(a, l, m)         # conquer (left half)
        merge_sort(a, m + 1, r)     # conquer (right half)
        merge(a, l, m, r)           # combine

# ---- variants that print the trace ----
def merge_sort_log(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort_log(a, l, m)
        merge_sort_log(a, m + 1, r)
        left, right = a[l:m + 1], a[m + 1:r + 1]
        merge(a, l, m, r)
        print(f"Merge(a,{l},{m},{r})  {left} and {right}  ->  {a}")

def merge_trace(a, l, m, r):
    n1 = m - l + 1
    n2 = r - m
    L = a[l:m + 1]
    R = a[m + 1:r + 1]
    print(f"Merge(a, {l}, {m}, {r}) on a = {a}")
    print(f"n1={n1}, n2={n2}, L={L}, R={R}")
    i = j = 0
    k = l
    it = 0
    while i < n1 and j < n2:
        it += 1
        head = f"{it:>2d}. loop 1: L[{i}]={L[i]} <= R[{j}]={R[j]}? "
        if L[i] <= R[j]:
            head += f"Yes => a[{k}] = L[{i}] = {L[i]}"
            a[k] = L[i]
            i += 1
        else:
            head += f"No => a[{k}] = R[{j}] = {R[j]}"
            a[k] = R[j]
            j += 1
        k += 1
        print(f"{head}, i={i}, j={j}, k={k}; a={a}")
    while i < n1:
        a[k] = L[i]
        print(f"    loop 2 (rest of L): a[{k}] = L[{i}] = {L[i]}; a={a}")
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        print(f"    loop 3 (rest of R): a[{k}] = R[{j}] = {R[j]}; a={a}")
        j += 1
        k += 1
    print(f"result a = {a}, comparisons = {it}")

a = [9, 12, 3, 16, 19, 11, 12]
n = len(a)
print("a =", a, "  n =", n)
print("Calls of Merge in execution order:")
merge_sort_log(a[:], 0, n - 1)
comparisons = 0
ms_calls = 0
merge_sort(a, 0, n - 1)
print(f"sorted = {a}   calls of MergeSort={ms_calls} (2n-1={2 * n - 1})   comparisons={comparisons}")
print()
merge_trace([5, 8, 2, 7], 0, 1, 3)

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Calls of Merge in execution order:
Merge(a,0,0,1)  [9] and [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] and [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] and [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] and [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] and [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] and [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
sorted = [3, 9, 11, 12, 12, 16, 19]   calls of MergeSort=13 (2n-1=13)   comparisons=14

Merge(a, 0, 1, 3) on a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. loop 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. loop 1: L[0]=5 <= R[1]=7? Yes => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. loop 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    loop 2 (rest of L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
result a = [2, 5, 7, 8], comparisons = 3


**Try it yourself.** Sort another array: the number of comparisons of `Merge` is at most $n\lceil\log_2 n\rceil$ and the calls of `MergeSort` are always $2n-1$.

In [4]:
# Sort another array and count the comparisons (at most n*ceil(log2 n)).
b = [38, 27, 43, 3, 9, 82, 10, 5]
comparisons = 0
ms_calls = 0
merge_sort(b, 0, len(b) - 1)
print(b, comparisons, ms_calls)

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: an array $a$ of natural numbers sorted in non-decreasing order, two indices $left$ and $right$ and a value $m$. *Output*: an index $i\in\{left,\dots,right\}$ with $a[i]=m$, or $-1$ if $m$ is not in $a[left..right]$. Every call makes **at most one** recursive call on (about) half of the interval: the call tree is a path, with at most $\lfloor\log_2 n\rfloor+2$ calls. To search the whole array we call `RecursiveBinarySearch(a, 0, n-1, m)`.

On the same array $a=[3,9,11,12,12,16,19]$ of the chapter we search $m=12,10,3,19$. For each search the table has one row per call ($left$, $right$, $mid$, $a[mid]$, action), as in the chapter but with indices from 0: adding 1 gives those of the chapter (for example $m=19$ is found at index 6, that is position 7).

In [5]:
rbs_calls = 0   # calls executed

def recursive_binary_search(a, left, right, m):
    """Index of m in a[left..right] (sorted array), or -1."""
    global rbs_calls
    rbs_calls += 1
    if left > right:
        return -1                   # empty interval: not found
    mid = left + (right - left) // 2
    if a[mid] == m:
        return mid                  # found
    if a[mid] < m:
        return recursive_binary_search(a, mid + 1, right, m)   # right half
    return recursive_binary_search(a, left, mid - 1, m)        # left half

def rbs_trace(a, left, right, m, k):
    """As above, but prints one row per call (k = number of the call)."""
    if left > right:
        print(f"{k:>4d} {left:>4d} {right:>5d} {'-':>3s} {'-':>6s} left > right => not found, return -1")
        return -1
    mid = left + (right - left) // 2
    row = f"{k:>4d} {left:>4d} {right:>5d} {mid:>3d} {a[mid]:>6d} "
    if a[mid] == m:
        print(row + f"a[mid] = m => found, return {mid}")
        return mid
    if a[mid] < m:
        print(row + f"a[mid] < m => call on left={mid + 1}, right={right}")
        return rbs_trace(a, mid + 1, right, m, k + 1)
    print(row + f"a[mid] > m => call on left={left}, right={mid - 1}")
    return rbs_trace(a, left, mid - 1, m, k + 1)

a = [3, 9, 11, 12, 12, 16, 19]
n = len(a)
print("a =", a, "  n =", n)
for m in [12, 10, 3, 19]:
    print()
    print(f"Search for m = {m}")
    print(f"{'call':>4s} {'left':>4s} {'right':>5s} {'mid':>3s} {'a[mid]':>6s} action")
    rbs_trace(a, 0, n - 1, m, 1)
print()
for m in [12, 10, 3, 19]:
    rbs_calls = 0
    r = recursive_binary_search(a, 0, n - 1, m)
    print(f"m={m:<3d} -> {r:>2d} ({r + 1 if r >= 0 else -1:>2d} with indices from 1)  calls={rbs_calls}")

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Search for m = 12
call left right mid a[mid] action
   1    0     6   3     12 a[mid] = m => found, return 3

Search for m = 10
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] < m => call on left=2, right=2
   3    2     2   2     11 a[mid] > m => call on left=2, right=1
   4    2     1   -      - left > right => not found, return -1

Search for m = 3
call left right mid a[mid] action
   1    0     6   3     12 a[mid] > m => call on left=0, right=2
   2    0     2   1      9 a[mid] > m => call on left=0, right=0
   3    0     0   0      3 a[mid] = m => found, return 0

Search for m = 19
call left right mid a[mid] action
   1    0     6   3     12 a[mid] < m => call on left=4, right=6
   2    4     6   5     16 a[mid] < m => call on left=6, right=6
   3    6     6   6     19 a[mid] = m => found, return 6

m=12  ->  3 ( 4 with indices from 1)  calls=1
m=10  -> -1 (

**Try it yourself.** Search another value (present or absent) and check that the calls are at most $\lfloor\log_2 n\rfloor+2$.

In [6]:
# Search another value, also an absent one: the calls are at most floor(log2 n) + 2.
rbs_calls = 0
print(recursive_binary_search(a, 0, len(a) - 1, 16), rbs_calls)
rbs_calls = 0
print(recursive_binary_search(a, 0, len(a) - 1, 100), rbs_calls)

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. It examines all the $n(n+1)/2$ subarrays $a[l..r]$ (non-empty, of consecutive elements), computing the sums incrementally: $\Theta(n^2)$.

On the example $a=[-2,5,-1,3]$ of the chapter the table has one row for each subarray examined ($l$, $r$, sum, action; every row that does not improve shows the current $best$). Indices are from 0: the chapter finds $(2,4,7)$, that is $a[2..4]=[5,-1,3]$ with sum $7$, here $(1,3,7)$.

In [7]:
def brute_force_maximum_subarray(a, n):
    """Returns (bl, br, best): a maximum-sum subarray of a[0..n-1]."""
    best = a[0]
    bl = 0
    br = 0
    l = 0
    while l < n:
        s = 0
        r = l
        while r < n:
            s = s + a[r]                  # sum of a[l..r]
            if s > best:
                best = s
                bl = l
                br = r
            r = r + 1
        l = l + 1
    return (bl, br, best)

def brute_force_trace(a, n):
    """As above, but prints one row for each subarray examined."""
    best = a[0]
    bl = 0
    br = 0
    for l in range(n):
        s = 0
        for r in range(l, n):
            s = s + a[r]
            if s > best:
                best, bl, br = s, l, r
                action = f"sum > best => best={best}, bl={bl}, br={br}"
            else:
                action = f"sum <= best={best}"
            print(f"{l:>3d} {r:>3d} {s:>5d}  {action}")
    return (bl, br, best)

a = [-2, 5, -1, 3]
n = len(a)
print("a =", a, "  n =", n, "  subarrays n(n+1)/2 =", n * (n + 1) // 2)
print(f"{'l':>3s} {'r':>3s} {'sum':>5s}  action")
bl, br, best = brute_force_trace(a, n)
print(f"result (bl, br, best) = ({bl}, {br}, {best})   ({bl + 1}, {br + 1}, {best} with indices from 1)")
print("BruteForceMaximumSubarray(a, n) =", brute_force_maximum_subarray(a, n))

a = [-2, 5, -1, 3]   n = 4   subarrays n(n+1)/2 = 10
  l   r   sum  action
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
result (bl, br, best) = (1, 3, 7)   (2, 4, 7 with indices from 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Try it yourself.** Change the array (for example with only negative numbers): the maximum subarray is the single largest element.

In [8]:
# Try an array of only negative numbers: the maximum subarray is the single largest element.
b = [-3, -1, -2, -5]
print(brute_force_maximum_subarray(b, len(b)))

(1, 1, -1)


## 5. FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [9]:
fms_calls = 0   # calls of FindMaximumSubarray

def find_max_crossing_subarray(a, l, mid, r):
    """Returns (maxleft, maxright, sum) of the best subarray crossing mid."""
    leftsum = float("-inf")
    s = 0
    maxleft = mid
    for i in range(mid, l - 1, -1):         # from mid down to l
        s = s + a[i]                        # sum of a[i..mid]
        if s > leftsum:
            leftsum = s
            maxleft = i
    rightsum = float("-inf")
    s = 0
    maxright = mid + 1
    for j in range(mid + 1, r + 1):         # from mid+1 to r
        s = s + a[j]                        # sum of a[mid+1..j]
        if s > rightsum:
            rightsum = s
            maxright = j
    return (maxleft, maxright, leftsum + rightsum)

def find_maximum_subarray(a, l, r):
    """Returns (i, j, sum) of a maximum-sum subarray of a[l..r]."""
    global fms_calls
    fms_calls += 1
    if l == r:
        return (l, r, a[l])                 # base case: one element
    mid = l + (r - l) // 2
    ll, lr, lsum = find_maximum_subarray(a, l, mid)
    rl, rr, rsum = find_maximum_subarray(a, mid + 1, r)
    cl, cr, csum = find_max_crossing_subarray(a, l, mid, r)
    if lsum >= rsum and lsum >= csum:
        return (ll, lr, lsum)
    if rsum >= lsum and rsum >= csum:
        return (rl, rr, rsum)
    return (cl, cr, csum)

# ---- variants that print the trace ----
def fms_trace(a, l, r, depth):
    pad = "  " * depth
    if l == r:
        print(f"{pad}FindMaximumSubarray(a, {l}, {r}) = ({l}, {r}, {a[l]})  base case")
        return (l, r, a[l])
    mid = l + (r - l) // 2
    print(f"{pad}FindMaximumSubarray(a, {l}, {r}): l != r, mid = {mid}")
    left = fms_trace(a, l, mid, depth + 1)
    right = fms_trace(a, mid + 1, r, depth + 1)
    cross = find_max_crossing_subarray(a, l, mid, r)
    print(f"{pad}  FindMaxCrossingSubarray(a, {l}, {mid}, {r}) = {cross}")
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    print(f"{pad}  left.sum={left[2]}, right.sum={right[2]}, cross.sum={cross[2]} => returns {ret}")
    return ret

def fms_rows(a, l, r, rows):
    if l == r:
        return (l, r, a[l])
    mid = l + (r - l) // 2
    left = fms_rows(a, l, mid, rows)
    right = fms_rows(a, mid + 1, r, rows)
    cross = find_max_crossing_subarray(a, l, mid, r)
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    rows.append((l, r, mid, left, right, cross, ret))
    return ret

def crossing_trace(a, l, mid, r):
    print(f"FindMaxCrossingSubarray(a, {l}, {mid}, {r})")
    print(f"{'i':>3s} {'a[i]':>5s} {'sum':>5s} {'leftsum':>8s} {'maxleft':>8s}")
    leftsum = float("-inf")
    s = 0
    for i in range(mid, l - 1, -1):
        s = s + a[i]
        if s > leftsum:
            leftsum, maxleft = s, i
        print(f"{i:>3d} {a[i]:>5d} {s:>5d} {leftsum:>8d} {maxleft:>8d}")
    print(f"{'j':>3s} {'a[j]':>5s} {'sum':>5s} {'rightsum':>8s} {'maxright':>8s}")
    rightsum = float("-inf")
    s = 0
    for j in range(mid + 1, r + 1):
        s = s + a[j]
        if s > rightsum:
            rightsum, maxright = s, j
        print(f"{j:>3d} {a[j]:>5d} {s:>5d} {rightsum:>8d} {maxright:>8d}")
    print(f"crossing subarray a[{maxleft}..{maxright}], sum {leftsum} + {rightsum} = {leftsum + rightsum}")

a = [-2, 5, -1, 3]
print("a =", a)
res = fms_trace(a, 0, len(a) - 1, 0)
fms_calls = 0
find_maximum_subarray(a, 0, len(a) - 1)
print(f"result {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} with indices from 1)   calls={fms_calls} (2n-1={2 * len(a) - 1})")

print()
a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
fms_calls = 0
res = find_maximum_subarray(a, 0, n - 1)
print(f"FindMaximumSubarray(a, 0, {n - 1}) = {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} with indices from 1)   calls={fms_calls} (2n-1={2 * n - 1})")
rows = []
fms_rows(a, 0, n - 1, rows)
rows.sort(key=lambda t: (-(t[1] - t[0]), t[0]))
print(f"{'call':<9s} {'mid':>3s}  {'left':<13s} {'right':<13s} {'crossing':<13s} returned")
for (l, r, mid, left, right, cross, ret) in rows:
    print(f"{'[' + str(l) + '..' + str(r) + ']':<9s} {mid:>3d}  {str(left):<13s} {str(right):<13s} {str(cross):<13s} {ret}")
print()
crossing_trace(a, 0, 3, 7)
print()
crossing_trace(a, 8, 11, 15)

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [10]:
# Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
b = [4, -1, 2, 1, -5, 4, -7, 3, 3]
fms_calls = 0
print(find_maximum_subarray(b, 0, len(b) - 1), fms_calls)

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n$ integers. *Output*: the indices $bl\le br$ of a maximum-sum subarray and the sum $best$. A single scan: $cur$ is the maximum sum of a subarray **ending** at the current position and $cl$ its first index; if $cur<0$ we restart from $a[i]$, otherwise we extend. $\Theta(n)$ steps, $\Theta(1)$ extra memory. (Chapter 18 has the compact version that returns only the sum.)

On the stock example (daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) we print the table of the chapter: one row per iteration, with the values at the end of the iteration (indices from 0, row $i=0$ is the initialization). The result $(7,10,43)$, that is $(8,11,43)$ with indices from 1, coincides with the one of the divide and conquer algorithm.

In [11]:
def kadane(a, n):
    """Returns (bl, br, best): a maximum-sum subarray of a[0..n-1]."""
    best = a[0]
    bl = 0
    br = 0
    cur = a[0]                  # maximum sum of a subarray ending at i
    cl = 0
    i = 1
    while i < n:
        if cur < 0:
            cur = a[i]          # restart
            cl = i
        else:
            cur = cur + a[i]    # extend
        if cur > best:
            best = cur
            bl = cl
            br = i
        i = i + 1
    return (bl, br, best)

def kadane_trace(a, n):
    """As above, but prints one row per iteration."""
    best, bl, br = a[0], 0, 0
    cur, cl = a[0], 0
    print(f"{0:>3d} {a[0]:>5d}  {'initialization':<24s} {cur:>4d} {cl:>3d} {best:>5d}  ({bl}, {br})")
    for i in range(1, n):
        if cur < 0:
            cur, cl = a[i], i
            action = "restart (cur<0)"
        else:
            cur = cur + a[i]
            action = "extend"
        if cur > best:
            best, bl, br = cur, cl, i
        print(f"{i:>3d} {a[i]:>5d}  {action:<24s} {cur:>4d} {cl:>3d} {best:>5d}  ({bl}, {br})")
    return (bl, br, best)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
print(f"{'i':>3s} {'a[i]':>5s}  {'action':<24s} {'cur':>4s} {'cl':>3s} {'best':>5s}  (bl, br)")
res = kadane_trace(a, n)
print(f"result {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} with indices from 1)")
b = [-2, 5, -1, 3]
print("a =", b, " Kadane(a, 4) =", kadane(b, len(b)))

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  action                    cur  cl  best  (bl, br)
  0    13  initialization             13   0    13  (0, 0)
  1    -3  extend                     10   0    13  (0, 0)
  2   -25  extend                    -15   0    13  (0, 0)
  3    20  restart (cur<0)            20   3    20  (3, 3)
  4    -3  extend                     17   3    20  (3, 3)
  5   -16  extend                      1   3    20  (3, 3)
  6   -23  extend                    -22   3    20  (3, 3)
  7    18  restart (cur<0)            18   7    20  (3, 3)
  8    20  extend                     38   7    38  (7, 8)
  9    -7  extend                     31   7    38  (7, 8)
 10    12  extend                     43   7    43  (7, 10)
 11    -5  extend                     38   7    43  (7, 10)
 12   -22  extend                     16   7    43  (7, 10)
 13    15  extend                     31   7    43  (7, 10)
 14    -4  extend                    

**Try it yourself.** Change the array (for example with only negative numbers): Kadane restarts at every step and finds the largest element.

In [12]:
# Try an array of only negative numbers: Kadane restarts at every step and finds the largest element.
c = [-3, -1, -2, -5]
print(kadane(c, len(c)))

(1, 1, -1)
